# Socratic-OT | Phase 2: Tutoring Policy — LangGraph State Machine

**Project:** Socratic-OT Multimodal AI Tutor  
**Team:** Vidhyadhari Bandaru, Richie Ilavarapu

### What this notebook does
1. Defines the **LangGraph dialogue state machine** (RAPPORT → HINT → CLUE → REVEAL → ASSESS)
2. Implements the **Tutoring Controller** — answer masking, escalation, stuck detection
3. Uses **Groq (Llama 3.1 8B)** as the primary LLM — free API, no credit card needed
4. Enforces **Socratic Purity**: no direct answer before Turn 3
5. Runs sample conversations to verify the purity constraint holds

### State machine overview
```
START
  │
  ▼
[RAPPORT]  →  Context-aware opening, build engagement
  │
  ▼
[HINT]     →  Turn 1: leading question, answer is MASKED
  │
  ├─ student correct ──────────────────────────────→ [ASSESS]
  ▼
[CLUE]     →  Turn 2: more specific clue, still NO direct answer
  │
  ├─ student correct ──────────────────────────────→ [ASSESS]
  ▼
[REVEAL]   →  Turn 3+: can now state answer + explanation
  │
  ▼
[ASSESS]   →  Clinical scenario prompt + mastery summary
```

### Prerequisite
Run `01_knowledge_base_chromadb.ipynb` first. This notebook loads the persisted ChromaDB.

---
## Step 0 — Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-groq chromadb sentence-transformers groq python-dotenv

---
## Step 1 — Mount Drive & Configure API Keys

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')

# ── Project paths ───────────────────────────────────────────────────────────
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/Socratic_OT'   # <-- adjust if needed
CHROMA_PERSIST     = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db')
IMAGE_META_JSON    = os.path.join(DRIVE_PROJECT_ROOT, 'Data/image_metadata/image_metadata.json')

# ── Groq API key ────────────────────────────────────────────────────────────
# Option A: Colab Secrets (recommended — click 🔑 in left sidebar, add GROQ_API_KEY)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ Groq key loaded from Colab Secrets')
except Exception:
    # Option B: Paste directly (less secure, don't commit)
    GROQ_API_KEY = 'gsk_YOUR_KEY_HERE'
    print('⚠️  Using hardcoded key — switch to Colab Secrets for safety')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('Setup complete.')

---
## Step 2 — Reload Retriever from Phase 1

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import torch, json, pickle

# Embedding model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f'Embedder loaded on {device}')

# ChromaDB
client     = chromadb.PersistentClient(path=CHROMA_PERSIST)
collection = client.get_collection('socratic_ot_textbook')
print(f'ChromaDB collection loaded: {collection.count()} chunks')

# Image metadata
with open(IMAGE_META_JSON) as f:
    image_metadata = json.load(f)

# Image lookup (saved in Phase 1)
lookup_path = os.path.join(CHROMA_PERSIST, 'image_lookup.pkl')
with open(lookup_path, 'rb') as f:
    img_lookup = pickle.load(f)
image_by_topic     = img_lookup['image_by_topic']
image_by_structure = img_lookup['image_by_structure']
print(f'Image metadata loaded: {len(image_metadata)} records')


def retrieve(query: str, top_k: int = 5) -> list:
    """Retrieve top-k relevant chunks for a query."""
    qemb = embedder.encode(query, normalize_embeddings=True).tolist()
    res  = collection.query(
        query_embeddings=[qemb], n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    hits = []
    for doc, meta, dist, cid in zip(
        res['documents'][0], res['metadatas'][0],
        res['distances'][0], res['ids'][0]
    ):
        hits.append({
            'chunk_id': cid,
            'score':    round(1 - dist, 4),
            'chapter':  meta.get('chapter', ''),
            'section':  meta.get('section', ''),
            'topic':    meta.get('topic', ''),
            'text':     doc
        })
    # Merge same-section chunks
    merged = {}
    for h in hits:
        key = h['section']
        if key not in merged:
            merged[key] = h.copy()
        else:
            if h['score'] > merged[key]['score']:
                merged[key]['score'] = h['score']
            if h['text'] not in merged[key]['text']:
                merged[key]['text'] += '\n\n' + h['text']
    return sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:3]


print('✅ Retriever ready')

---
## Step 3 — Configure Groq LLM

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Primary model: Llama 3.1 8B on Groq (as committed in the presentation)
llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.4,          # low temp → consistent, grounded responses
    max_tokens=512,
    api_key=GROQ_API_KEY
)

# Quick smoke test
test_resp = llm.invoke([HumanMessage(content='Say OK in one word.')])
print(f'✅ Groq LLM connected: {test_resp.content.strip()}')

---
## Step 4 — Define Dialogue State Schema

LangGraph passes a typed state dict between nodes.

In [ ]:
from typing import TypedDict, Annotated, Optional
import operator

class DialogueState(TypedDict):
    # ── Core conversation ────────────────────────────────────────────────────
    messages        : Annotated[list, operator.add]   # full chat history
    current_input   : str                              # latest student message

    # ── Dialogue control ─────────────────────────────────────────────────────
    phase           : str    # RAPPORT | HINT | CLUE | REVEAL | ASSESS | DONE
    turn_in_phase   : int    # how many turns have happened in current phase
    total_turns     : int    # total conversation turns

    # ── Retrieval context ────────────────────────────────────────────────────
    retrieved_chunks: list   # top chunks for current topic
    masked_answer   : str    # the answer we are hiding from the student
    topic_label     : str    # inferred topic of the question

    # ── Session memory ───────────────────────────────────────────────────────
    weak_topics     : list   # topics the student has struggled with
    wrong_guesses   : list   # incorrect answers given this session
    correct_count   : int    # correct turns this session
    stuck_count     : int    # consecutive wrong attempts

    # ── Output ───────────────────────────────────────────────────────────────
    tutor_response  : str    # the tutor's response text
    next_node       : str    # routing hint for conditional edges

print('✅ DialogueState schema defined')

---
## Step 5 — System Prompts for Each Phase

In [ ]:
SYSTEM_BASE = """You are Socratic-OT, a Socratic tutor for Occupational Therapy students.
You help students master Gross Anatomy and Neuroscience through guided questioning,
NOT direct answers. All your responses must be grounded in the provided textbook context.
Never cite sources you were not given. Never fabricate anatomical facts."""

# ── RAPPORT: warm opening ────────────────────────────────────────────────────
RAPPORT_PROMPT = SYSTEM_BASE + """

PHASE: RAPPORT
You are in the opening phase. Your goal is to build engagement with a brief,
context-aware greeting. Reference the topic the student mentioned.
Ask ONE warm opening question to get them thinking — do NOT start tutoring yet.
Keep your response to 2-3 sentences maximum."""

# ── HINT: Turn 1 — NO ANSWER ────────────────────────────────────────────────
HINT_PROMPT = SYSTEM_BASE + """

PHASE: HINT (Turn 1)
STRICT RULE: You are FORBIDDEN from stating the answer, term, or definition directly.
This is the first tutoring turn. You must:
  1. Use the TEXTBOOK CONTEXT below to identify the target concept
  2. Ask ONE leading question that makes the student recall it from memory
  3. Your question must reference a related fact FROM the context (function, location, clinical sign)
  4. Do NOT name the target term. Do NOT define it. Do NOT say 'it starts with...'

TEXTBOOK CONTEXT:
{context}

TARGET ANSWER (MASKED — never reveal this word/phrase in this turn): {masked_answer}"""

# ── CLUE: Turn 2 — STILL NO ANSWER ─────────────────────────────────────────
CLUE_PROMPT = SYSTEM_BASE + """

PHASE: CLUE (Turn 2)
STRICT RULE: You still cannot state the final answer directly.
The student tried: "{student_attempt}"
Your response must:
  1. Acknowledge what was correct in their attempt (if anything)
  2. Give a MORE specific hint — use the context to narrow it down
  3. Still NOT name the term outright
  4. Ask a follow-up question to guide the final step

TEXTBOOK CONTEXT:
{context}

TARGET ANSWER (MASKED): {masked_answer}"""

# ── REVEAL: Turn 3+ ─────────────────────────────────────────────────────────
REVEAL_PROMPT = SYSTEM_BASE + """

PHASE: REVEAL (Turn 3+ or student is stuck)
The student has had two guided attempts. You may now:
  1. Confirm or gently correct their answer
  2. State the correct term/concept clearly
  3. Give a grounded explanation using the TEXTBOOK CONTEXT
  4. End with a brief clinical connection relevant to OT

TEXTBOOK CONTEXT:
{context}

CORRECT ANSWER: {masked_answer}"""

# ── ASSESS: Clinical application ────────────────────────────────────────────
ASSESS_PROMPT = SYSTEM_BASE + """

PHASE: ASSESSMENT
The student now knows the concept. Move beyond recall into clinical reasoning.
Present a brief clinical OT scenario that requires applying this concept.
Ask the student to reason through the scenario — do NOT give the answer yet.
Keep the scenario realistic and relevant to NBCOT-level clinical reasoning.

CONCEPT: {masked_answer}
TEXTBOOK CONTEXT:
{context}"""

# ── MASTERY SUMMARY ─────────────────────────────────────────────────────────
SUMMARY_PROMPT = SYSTEM_BASE + """

PHASE: MASTERY SUMMARY
The tutoring session on this concept is complete.
Write a short mastery summary covering:
  1. The core concept ({masked_answer})
  2. What the student demonstrated correctly
  3. Any gaps or misconceptions to revisit
  4. A one-sentence clinical takeaway for OT practice
Keep it to 5-6 sentences.

Student's attempt history: {attempt_history}
TEXTBOOK CONTEXT: {context}"""

print('✅ All phase prompts defined')

---
## Step 6 — Core LangGraph Nodes

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: call LLM with a given system prompt + history
# ─────────────────────────────────────────────────────────────────────────────
def call_llm(system_prompt: str, history: list, user_msg: str = '') -> str:
    msgs = [SystemMessage(content=system_prompt)]
    for m in history[-6:]:   # keep last 6 turns to avoid token overflow
        msgs.append(m)
    if user_msg:
        msgs.append(HumanMessage(content=user_msg))
    resp = llm.invoke(msgs)
    return resp.content.strip()


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: extract the masked answer from retrieved context
# ─────────────────────────────────────────────────────────────────────────────
def extract_masked_answer(query: str, context_text: str) -> str:
    """Ask the LLM to identify the core answer, which we then hide from the student."""
    prompt = f"""Given this anatomy/neuroscience question and textbook context,
identify the single most important term, structure, or concept the student is asking about.
Return ONLY that term or phrase (2-5 words max). Nothing else.

Question: {query}
Context: {context_text[:800]}"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip()


# ─────────────────────────────────────────────────────────────────────────────
# NODE 1: ROUTER — classify input, retrieve context, extract masked answer
# ─────────────────────────────────────────────────────────────────────────────
def router_node(state: DialogueState) -> DialogueState:
    """
    Entry point. Classifies the student's input and retrieves relevant context.
    Sets the next_node for routing.
    """
    user_input = state['current_input']
    phase      = state.get('phase', 'RAPPORT')

    # Retrieve context for the student's query
    chunks = retrieve(user_input, top_k=5)
    context_text = '\n\n'.join([c['text'] for c in chunks]) if chunks else ''

    # Extract masked answer on first tutoring turn
    masked = state.get('masked_answer', '')
    if not masked and phase in ('HINT', 'RAPPORT'):
        masked = extract_masked_answer(user_input, context_text)

    # Infer topic label
    topic = chunks[0]['topic'] if chunks else 'anatomy'

    return {
        **state,
        'retrieved_chunks': chunks,
        'masked_answer':    masked,
        'topic_label':      topic,
        'next_node':        phase
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 2: RAPPORT — warm opening
# ─────────────────────────────────────────────────────────────────────────────
def rapport_node(state: DialogueState) -> DialogueState:
    user_input = state['current_input']
    response   = call_llm(RAPPORT_PROMPT, state['messages'], user_input)

    new_messages = state['messages'] + [
        HumanMessage(content=user_input),
        AIMessage(content=response)
    ]
    return {
        **state,
        'messages':       new_messages,
        'tutor_response': response,
        'phase':          'HINT',          # move to tutoring next turn
        'turn_in_phase':  0,
        'total_turns':    state.get('total_turns', 0) + 1
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 3: HINT — Turn 1, NO answer leak allowed
# ─────────────────────────────────────────────────────────────────────────────
def hint_node(state: DialogueState) -> DialogueState:
    user_input   = state['current_input']
    context_text = '\n\n'.join([c['text'] for c in state['retrieved_chunks']])
    masked       = state['masked_answer']

    sys_prompt = HINT_PROMPT.format(context=context_text[:2000], masked_answer=masked)
    response   = call_llm(sys_prompt, state['messages'], user_input)

    # ── Socratic Purity Guard ──────────────────────────────────────────────
    # Double-check: if the masked answer leaked, regenerate with stricter instruction
    if masked.lower() in response.lower():
        stricter = sys_prompt + '\n\nCRITICAL: Your previous response contained the masked answer. Rewrite WITHOUT mentioning it at all.'
        response = call_llm(stricter, state['messages'], user_input)

    new_messages = state['messages'] + [
        HumanMessage(content=user_input),
        AIMessage(content=response)
    ]
    return {
        **state,
        'messages':       new_messages,
        'tutor_response': response,
        'phase':          'CLUE',
        'turn_in_phase':  1,
        'total_turns':    state.get('total_turns', 0) + 1
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 4: CLUE — Turn 2, still NO direct answer
# ─────────────────────────────────────────────────────────────────────────────
def clue_node(state: DialogueState) -> DialogueState:
    user_input   = state['current_input']
    context_text = '\n\n'.join([c['text'] for c in state['retrieved_chunks']])
    masked       = state['masked_answer']
    stuck        = state.get('stuck_count', 0)

    # Check if student got it right in their HINT response
    if masked.lower() in user_input.lower():
        # Student got it — skip to REVEAL then ASSESS
        return {
            **state,
            'phase':         'REVEAL',
            'current_input': user_input,
            'correct_count': state.get('correct_count', 0) + 1
        }

    # Student didn't get it — give clue
    sys_prompt = CLUE_PROMPT.format(
        context=context_text[:2000],
        masked_answer=masked,
        student_attempt=user_input
    )
    response = call_llm(sys_prompt, state['messages'], user_input)

    # Purity guard
    if masked.lower() in response.lower():
        stricter = sys_prompt + '\n\nCRITICAL: Do NOT state the answer. Rephrase the clue without naming it.'
        response = call_llm(stricter, state['messages'], user_input)

    # Track wrong guess
    wrong_guesses = state.get('wrong_guesses', []) + [user_input]

    new_messages = state['messages'] + [
        HumanMessage(content=user_input),
        AIMessage(content=response)
    ]
    return {
        **state,
        'messages':      new_messages,
        'tutor_response': response,
        'phase':         'REVEAL',
        'turn_in_phase': 2,
        'total_turns':   state.get('total_turns', 0) + 1,
        'stuck_count':   stuck + 1,
        'wrong_guesses': wrong_guesses
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 5: REVEAL — Turn 3+, can now state the answer
# ─────────────────────────────────────────────────────────────────────────────
def reveal_node(state: DialogueState) -> DialogueState:
    user_input   = state['current_input']
    context_text = '\n\n'.join([c['text'] for c in state['retrieved_chunks']])
    masked       = state['masked_answer']

    sys_prompt = REVEAL_PROMPT.format(context=context_text[:2000], masked_answer=masked)
    response   = call_llm(sys_prompt, state['messages'], user_input)

    new_messages = state['messages'] + [
        HumanMessage(content=user_input),
        AIMessage(content=response)
    ]

    # Update weak topics if student needed full reveal
    weak = state.get('weak_topics', [])
    topic = state.get('topic_label', '')
    if state.get('stuck_count', 0) > 0 and topic and topic not in weak:
        weak = weak + [topic]

    return {
        **state,
        'messages':       new_messages,
        'tutor_response': response,
        'phase':          'ASSESS',
        'total_turns':    state.get('total_turns', 0) + 1,
        'weak_topics':    weak
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 6: ASSESS — clinical scenario
# ─────────────────────────────────────────────────────────────────────────────
def assess_node(state: DialogueState) -> DialogueState:
    context_text = '\n\n'.join([c['text'] for c in state['retrieved_chunks']])
    masked       = state['masked_answer']

    sys_prompt = ASSESS_PROMPT.format(context=context_text[:2000], masked_answer=masked)
    response   = call_llm(sys_prompt, state['messages'])

    new_messages = state['messages'] + [AIMessage(content=response)]
    return {
        **state,
        'messages':       new_messages,
        'tutor_response': response,
        'phase':          'DONE',
        'total_turns':    state.get('total_turns', 0) + 1
    }


print('✅ All LangGraph nodes defined')

---
## Step 7 — Build the LangGraph State Machine

In [ ]:
from langgraph.graph import StateGraph, END

# ── Routing function (decides which node to call next) ──────────────────────
def route(state: DialogueState) -> str:
    """Conditional edge: reads state.phase and routes to the correct node."""
    phase = state.get('phase', 'RAPPORT')
    mapping = {
        'RAPPORT': 'rapport',
        'HINT':    'hint',
        'CLUE':    'clue',
        'REVEAL':  'reveal',
        'ASSESS':  'assess',
        'DONE':    END
    }
    return mapping.get(phase, END)


# ── Build graph ──────────────────────────────────────────────────────────────
builder = StateGraph(DialogueState)

# Add nodes
builder.add_node('router',  router_node)
builder.add_node('rapport', rapport_node)
builder.add_node('hint',    hint_node)
builder.add_node('clue',    clue_node)
builder.add_node('reveal',  reveal_node)
builder.add_node('assess',  assess_node)

# Entry point is always router
builder.set_entry_point('router')

# Router dispatches to the phase-specific node
builder.add_conditional_edges('router', route)

# Each phase node loops back to router for the next turn
for node in ['rapport', 'hint', 'clue', 'reveal']:
    builder.add_edge(node, END)   # Each turn = one graph invocation
builder.add_edge('assess', END)

# Compile
graph = builder.compile()

print('✅ LangGraph state machine compiled')
print('   Nodes:', ['router', 'rapport', 'hint', 'clue', 'reveal', 'assess'])
print('   Phase flow: RAPPORT → HINT → CLUE → REVEAL → ASSESS → DONE')

---
## Step 8 — Tutor Chat Session Manager

Wraps the graph with a session object for multi-turn conversations.

In [ ]:
class SocraticOTSession:
    """
    Manages a single tutoring session.
    Persists state between turns and exposes a simple chat() interface.
    """

    def __init__(self):
        self.state: DialogueState = {
            'messages':        [],
            'current_input':   '',
            'phase':           'RAPPORT',
            'turn_in_phase':   0,
            'total_turns':     0,
            'retrieved_chunks': [],
            'masked_answer':   '',
            'topic_label':     '',
            'weak_topics':     [],
            'wrong_guesses':   [],
            'correct_count':   0,
            'stuck_count':     0,
            'tutor_response':  '',
            'next_node':       ''
        }

    def chat(self, user_input: str) -> str:
        """
        Send one student message and get the tutor's response.
        Updates internal state automatically.
        """
        self.state['current_input'] = user_input
        result = graph.invoke(self.state)
        self.state = result
        return result['tutor_response']

    def get_session_summary(self) -> dict:
        """Return a summary of the current session for memory/logging."""
        return {
            'phase':          self.state['phase'],
            'total_turns':    self.state['total_turns'],
            'masked_answer':  self.state['masked_answer'],
            'weak_topics':    self.state['weak_topics'],
            'wrong_guesses':  self.state['wrong_guesses'],
            'correct_count':  self.state['correct_count']
        }

    def reset_for_new_topic(self):
        """Start a new Socratic loop for a new concept, keeping session memory."""
        prev_memory = {
            'weak_topics':  self.state['weak_topics'],
            'correct_count': self.state['correct_count']
        }
        self.__init__()
        self.state.update(prev_memory)
        self.state['phase'] = 'HINT'   # skip rapport on subsequent topics


print('✅ SocraticOTSession class defined')

---
## Step 9 — Run a Full Socratic Conversation

Demonstrate the complete dialogue flow. Verify Socratic purity (no answer leak before Turn 3).

In [ ]:
def run_demo_conversation(topic_question: str, student_turns: list):
    """
    Run a simulated conversation and print each turn with phase labels.
    student_turns: list of student replies to give after the first question
    """
    session = SocraticOTSession()
    print('═' * 70)
    print(f'TOPIC: {topic_question}')
    print('═' * 70)

    # Turn 0: student asks the question → rapport
    print(f'\n[Phase: RAPPORT | Turn 0]')
    print(f'Student: {topic_question}')
    response = session.chat(topic_question)
    print(f'Tutor: {response}')
    print(f'  >> Phase now: {session.state["phase"]} | Masked answer: "{session.state["masked_answer"]}"')

    # Subsequent student turns
    for i, student_msg in enumerate(student_turns, 1):
        print(f'\n[Phase: {session.state["phase"]} | Turn {i}]')
        print(f'Student: {student_msg}')
        response = session.chat(student_msg)
        print(f'Tutor: {response}')
        summary = session.get_session_summary()
        print(f'  >> Phase now: {summary["phase"]} | Stuck count: {summary["wrong_guesses"]}')  

    print('\n--- Session Summary ---')
    for k, v in session.get_session_summary().items():
        print(f'  {k}: {v}')


# ── Conversation 1: Median Nerve (from the presentation example) ─────────────
run_demo_conversation(
    topic_question='I need help understanding the nerve that controls thumb movement at the wrist.',
    student_turns=[
        'Is it the ulnar nerve?',                           # Turn 1 (HINT): wrong
        'Maybe the radial nerve? It goes through the wrist', # Turn 2 (CLUE): wrong
        'Oh wait — the median nerve!'                        # Turn 3 (REVEAL): correct
    ]
)

In [ ]:
# ── Conversation 2: Student gets it early (at Turn 1) ───────────────────────
run_demo_conversation(
    topic_question='Can you help me review how skeletal muscle contracts?',
    student_turns=[
        'The sarcomere shortens when actin and myosin slide together',  # Turn 1: correct
        'The calcium binds to troponin and exposes actin binding sites'  # Turn 2: assessment response
    ]
)

In [ ]:
# ── Socratic Purity Verification ─────────────────────────────────────────────
# Automated check: scan HINT and CLUE responses for the masked answer

def verify_socratic_purity(session_log: list, masked_answer: str) -> bool:
    """
    Returns True if the bot did NOT leak the answer in turns 1 or 2.
    session_log: list of (phase, tutor_response) tuples
    """
    ans_lower = masked_answer.lower()
    leaks = []
    for phase, response in session_log:
        if phase in ('HINT', 'CLUE') and ans_lower in response.lower():
            leaks.append((phase, response[:100]))

    if leaks:
        print(f'❌ PURITY VIOLATION — answer leaked in phases: {[l[0] for l in leaks]}')
        for phase, snippet in leaks:
            print(f'   [{phase}]: ...{snippet}...')
        return False
    else:
        print(f'✅ PURITY PASSED — "{masked_answer}" not revealed before Turn 3')
        return True


# Run a purity-test session and collect the log
purity_session = SocraticOTSession()
log = []

queries = [
    'Help me understand the brachial plexus',
    'Is it a cranial nerve?',
    'Something in the neck or shoulder region?',
    'The brachial plexus!'
]

for q in queries:
    resp = purity_session.chat(q)
    phase_used = 'HINT' if purity_session.state['total_turns'] == 2 else \
                 'CLUE' if purity_session.state['total_turns'] == 3 else \
                 purity_session.state['phase']
    log.append((phase_used, resp))

masked = purity_session.state['masked_answer']
print(f'Masked answer was: "{masked}"')
verify_socratic_purity(log[:3], masked)   # check first 3 turns only

---
## Step 10 — Generate 5 Transcript Logs for Evaluation

Required by the project rubric: 5 conversation transcripts showing no answer leak before Turn 3.

In [ ]:
import json, os

TRANSCRIPTS_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'Evaluation/transcripts')
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

# Five distinct topics spanning multiple chapters
transcript_scenarios = [
    {
        'id': 'T001',
        'question': 'Help me understand the nerve in the carpal tunnel',
        'student_turns': ['Ulnar nerve?', 'Radial nerve I think', 'Oh — median nerve']
    },
    {
        'id': 'T002',
        'question': 'What part of the brain controls voluntary movement?',
        'student_turns': ['Cerebellum?', 'The frontal lobe?', 'Primary motor cortex!']
    },
    {
        'id': 'T003',
        'question': 'Tell me about the muscle that flexes the elbow',
        'student_turns': ['Triceps?', 'Deltoid maybe?', 'Biceps brachii']
    },
    {
        'id': 'T004',
        'question': 'What type of joint is the hip joint?',
        'student_turns': ['Hinge joint?', 'Pivot joint?', 'Ball and socket joint']
    },
    {
        'id': 'T005',
        'question': 'How do neurons communicate across a synapse?',
        'student_turns': ['Through electrical signals directly?', 'Enzymes?', 'Neurotransmitters released into the synaptic cleft']
    }
]

all_purity_results = []

for scenario in transcript_scenarios:
    session = SocraticOTSession()
    transcript = {'id': scenario['id'], 'topic_question': scenario['question'], 'turns': []}
    log = []

    # First turn
    resp = session.chat(scenario['question'])
    transcript['turns'].append({'role': 'student', 'text': scenario['question'], 'phase': 'RAPPORT'})
    transcript['turns'].append({'role': 'tutor', 'text': resp, 'phase': 'RAPPORT'})

    # Student follow-ups
    phases_used = ['HINT', 'CLUE', 'REVEAL']
    for i, student_msg in enumerate(scenario['student_turns']):
        phase_label = phases_used[i] if i < len(phases_used) else 'POST'
        resp = session.chat(student_msg)
        transcript['turns'].append({'role': 'student', 'text': student_msg, 'phase': phase_label})
        transcript['turns'].append({'role': 'tutor', 'text': resp, 'phase': phase_label})
        if phase_label in ('HINT', 'CLUE'):
            log.append((phase_label, resp))

    masked = session.state['masked_answer']
    transcript['masked_answer']  = masked
    transcript['session_summary'] = session.get_session_summary()

    # Save transcript
    fpath = os.path.join(TRANSCRIPTS_DIR, f'{scenario["id"]}_transcript.json')
    with open(fpath, 'w') as f:
        json.dump(transcript, f, indent=2)

    # Verify purity
    passed = verify_socratic_purity(log, masked) if masked else True
    all_purity_results.append({'id': scenario['id'], 'passed': passed, 'masked_answer': masked})
    print(f'  Saved: {fpath}')

print()
print('=== SOCRATIC PURITY SUMMARY ===')
purity_score = sum(r['passed'] for r in all_purity_results)
for r in all_purity_results:
    status = '✅' if r['passed'] else '❌'
    print(f'  {status} {r["id"]} — Masked: "{r["masked_answer"]}')
print(f'\nSocratic Purity Score: {purity_score}/{len(all_purity_results)}')
print('Target: 5/5 (no answer leak before Turn 3)')

---
## Summary

In [ ]:
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('PHASE 2 COMPLETE')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  LLM               : Groq Llama 3.1 8B Instruct')
print('  Orchestration     : LangGraph state machine')
print('  Phases            : RAPPORT → HINT → CLUE → REVEAL → ASSESS')
print('  Socratic Purity   : Enforced (automated guard + regeneration)')
print('  Transcripts saved :', TRANSCRIPTS_DIR)
print()
print('→ Next: Run 03_session_memory.ipynb')